# 01.5 Sample-frequency filtered VAF and AltDepth KDE plots

This notebook calculates variant frequency across samples, removes variants seen in more than the selected sample-frequency threshold, then plots VAF and smoothed AltDepth KDE curves.

## 1. Settings

Edit the input file, output folder, sample-frequency thresholds, and AltDepth thresholds here.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

run_label = "Run2"

input_csv = Path(
    "/home/donetski/Notebooks/OutputFiles/04_qc_checking_on_target/04_Run2_per_variant_target_status_FULL.csv"
)

output_dir = Path(
    "/home/donetski/Notebooks/OutputFiles/01.5_sample_frequency_filtered_vaf_altdepth_kde"
) / run_label.lower()

output_prefix = "01.5"

filter_to_pass = True
filter_to_on_target = True

callers = ["DeepVariant", "Mutect2"]
variant_cols = ["Chr", "Start", "REF", "ALT"]

# Variants with sample_fraction > threshold are removed.
# 0.10 means remove variants seen in more than 10% of samples.
sample_frequency_thresholds = [0.10]

# Plots are made separately after each AltDepth cutoff.
altdepth_thresholds = [None, 1, 3, 5, 7, 10]

# Keeps AltDepth KDE plots readable. Set to None to show the full range.
altdepth_kde_xmax = 50

# Protects the kernel from very large KDE plots. Set to None if not needed.
max_kde_points = 100_000

TARGET_GENES = [
    "ATM", "BARD1", "BRCA1", "BRCA2", "CDH1", "CDKN2A", "CHEK2", "MLH1",
    "MSH2", "MSH6", "PALB2", "PMS2", "PTEN", "RAD51C", "RAD51D", "TP53",
]

## 2. Helper functions

These functions keep the rest of the notebook shorter: they check columns, save plots, calculate sample frequency, and draw KDE curves.

In [ ]:
def read_csv_auto(path):
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, low_memory=False, encoding="latin1")


def require_columns(df, columns):
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def safe_name(value):
    return (
        str(value)
        .replace(" ", "_")
        .replace("/", "_")
        .replace(".", "p")
        .replace("<=", "le")
        .replace(">=", "ge")
        .replace(">", "gt")
        .lower()
    )


def threshold_label(value):
    return str(value).replace(".", "p")


def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.show()
    print("Saved:", path)


def add_sample_frequency_columns(input_df):
    n_samples = input_df["Sample.ID"].nunique()

    one_row_per_sample_variant = input_df.drop_duplicates(
        subset=["Sample.ID"] + variant_cols
    )

    frequency_df = (
        one_row_per_sample_variant
        .groupby(variant_cols)
        .agg(sample_count=("Sample.ID", "nunique"))
        .reset_index()
    )

    frequency_df["total_samples_in_run"] = n_samples
    frequency_df["sample_fraction"] = frequency_df["sample_count"] / n_samples
    frequency_df["sample_percent"] = (frequency_df["sample_fraction"] * 100).round(2)

    return input_df.merge(frequency_df, on=variant_cols, how="left")


def plot_kde(series, label):
    values = pd.to_numeric(series, errors="coerce").dropna()

    if len(values) < 2 or values.nunique() < 2:
        return False

    if max_kde_points is not None and len(values) > max_kde_points:
        values = values.sample(max_kde_points, random_state=1)

    values.plot(kind="density", label=label)
    return True

## 3. Load data and apply base QC filters

This keeps the same basic filtering idea as the earlier VAF notebook: optional `PASS` variants and optional on-target variants.

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)

required_columns = [
    "Sample.ID", "caller", "Gene", "Sample.AltFrac", "Sample.AltDepth", "Sample.Depth"
] + variant_cols

df = read_csv_auto(input_csv)
require_columns(df, required_columns)

df["Sample.AltFrac"] = pd.to_numeric(df["Sample.AltFrac"], errors="coerce")
df["Sample.AltDepth"] = pd.to_numeric(df["Sample.AltDepth"], errors="coerce")
df["Sample.Depth"] = pd.to_numeric(df["Sample.Depth"], errors="coerce")

df_qc = df.copy()

if filter_to_pass:
    require_columns(df_qc, ["FILTER"])
    df_qc = df_qc[df_qc["FILTER"].eq("PASS")].copy()

if filter_to_on_target:
    require_columns(df_qc, ["on_target"])
    on_target = df_qc["on_target"].astype(str).str.lower().isin(["true", "1", "yes"])
    df_qc = df_qc[on_target].copy()

print("Input rows:", f"{len(df):,}")
print("Rows after base QC filters:", f"{len(df_qc):,}")
print("Samples after base QC filters:", f"{df_qc['Sample.ID'].nunique():,}")

## 4. Calculate sample frequency

Frequency is calculated after the base QC filters. Duplicate calls of the same variant in the same sample are counted once.

In [ ]:
df_qc = add_sample_frequency_columns(df_qc)

with_frequency_csv = output_dir / f"{output_prefix}_{run_label}_with_sample_frequency.csv"
df_qc.to_csv(with_frequency_csv, index=False)

print("Saved:", with_frequency_csv)
df_qc[["Sample.ID", "caller", "Gene"] + variant_cols + [
    "sample_count", "total_samples_in_run", "sample_fraction", "sample_percent"
]].head()

## 5. Plot functions

Each plot uses the sample-frequency-filtered data, then overlays curves for `AltDepth >= 1, 3, 5, 7, 10`.

In [ ]:
def plot_vaf_density_by_altdepth(caller_df, caller_name, set_name, sample_freq_threshold, figure_dir):
    plt.figure(figsize=(8, 5))

    plotted_any = False
    for altdepth_threshold in altdepth_thresholds:
        if altdepth_threshold is None:
            subset = caller_df.copy()
        else:
            subset = caller_df[caller_df["Sample.AltDepth"] >= altdepth_threshold].copy()
            
        label = f"AltDepth >= {altdepth_threshold} (n={len(subset):,})"
        plotted_any |= plot_kde(subset["Sample.AltFrac"], label)

    if not plotted_any:
        plt.close()
        print(f"Skipped VAF KDE: {set_name}, {caller_name}, sample_frequency <= {sample_freq_threshold}")
        return

    plt.axvline(0.5, linestyle="--", label="VAF ~0.5")
    plt.axvline(1.0, linestyle="--", label="VAF ~1.0")
    plt.xlim(0, 1.05)
    plt.xlabel("Sample.AltFrac")
    plt.ylabel("Density")
    plt.title(
        f"{run_label} {set_name}: {caller_name} VAF KDE\n"
        f"sample frequency <= {sample_freq_threshold:.0%}"
    )
    plt.legend(fontsize=8)

    out_png = figure_dir / (
        f"{output_prefix}_{run_label}_{set_name}_{safe_name(caller_name)}"
        f"_samplefreq_le_{threshold_label(sample_freq_threshold)}_vaf_kde_by_altdepth.png"
    )
    save_plot(out_png)


def plot_altdepth_kde_by_altdepth(caller_df, caller_name, set_name, sample_freq_threshold, figure_dir):
    plt.figure(figsize=(8, 5))

    plotted_any = False
    for altdepth_threshold in altdepth_thresholds:
        if altdepth_threshold is None:
            subset = caller_df.copy()
            label = f"No AltDepth cutoff (n={len(subset):,})"
        else:
            subset = caller_df[caller_df["Sample.AltDepth"] >= altdepth_threshold].copy()
            label = f"AltDepth >= {altdepth_threshold} (n={len(subset):,})"
    
        values = subset["Sample.AltDepth"]
    
        if altdepth_kde_xmax is not None:
            values = values[values <= altdepth_kde_xmax]
    
        if altdepth_threshold is None:
            plotted_any |= plot_kde(values, label)
            plt.gca().lines[-1].set_linestyle("--")
            plt.gca().lines[-1].set_linewidth(2.5)
        else:
            plotted_any |= plot_kde(values, label)
                
                
    if not plotted_any:
        plt.close()
        print(f"Skipped AltDepth KDE: {set_name}, {caller_name}, sample_frequency <= {sample_freq_threshold}")
        return

    if altdepth_kde_xmax is not None:
        plt.xlim(0, altdepth_kde_xmax)

    plt.xlabel("Sample.AltDepth")
    plt.ylabel("Density")
    plt.title(
        f"{run_label} {set_name}: {caller_name} AltDepth KDE\n"
        f"sample frequency <= {sample_freq_threshold:.0%}"
    )
    plt.legend(fontsize=8)

    out_png = figure_dir / (
        f"{output_prefix}_{run_label}_{set_name}_{safe_name(caller_name)}"
        f"_samplefreq_le_{threshold_label(sample_freq_threshold)}_altdepth_kde_by_altdepth.png"
    )
    save_plot(out_png)

## 6. Filter by sample frequency and plot

This loops through each sample-frequency threshold and each AltDepth threshold. Variants with `sample_fraction > threshold` are removed.

In [ ]:
target_gene_mask = df_qc["Gene"].astype(str).str.upper().isin(TARGET_GENES)

analysis_sets = {
    "all_variants": df_qc.copy(),
    "target_16_genes": df_qc[target_gene_mask].copy(),
}

summary_rows = []

for set_name, set_df in analysis_sets.items():
    for sample_freq_threshold in sample_frequency_thresholds:
        sample_freq_dir = output_dir / set_name / f"samplefreq_le_{threshold_label(sample_freq_threshold)}"
        csv_dir = sample_freq_dir / "csv"
        figure_dir = sample_freq_dir / "figures"

        csv_dir.mkdir(parents=True, exist_ok=True)
        figure_dir.mkdir(parents=True, exist_ok=True)

        kept_df = set_df[set_df["sample_fraction"] <= sample_freq_threshold].copy()
        removed_df = set_df[set_df["sample_fraction"] > sample_freq_threshold].copy()

        kept_csv = csv_dir / (
            f"{output_prefix}_{run_label}_{set_name}"
            f"_samplefreq_le_{threshold_label(sample_freq_threshold)}_kept.csv"
        )
        removed_csv = csv_dir / (
            f"{output_prefix}_{run_label}_{set_name}"
            f"_samplefreq_gt_{threshold_label(sample_freq_threshold)}_removed.csv"
        )

        kept_df.to_csv(kept_csv, index=False)
        removed_df.to_csv(removed_csv, index=False)

        print(f"\n{set_name}, sample_frequency <= {sample_freq_threshold:.0%}")
        print("Rows kept:", f"{len(kept_df):,}")
        print("Rows removed:", f"{len(removed_df):,}")
        print("Saved:", kept_csv)
        print("Saved:", removed_csv)

        for caller_name in callers:
            before_samplefreq_df = set_df[set_df["caller"].eq(caller_name)].copy()
            caller_df = kept_df[kept_df["caller"].eq(caller_name)].copy()
        
            # Compare AltDepth KDE before vs after sample-frequency filtering.
            plt.figure(figsize=(8, 5))
        
            plotted_any = False
        
            before_values = before_samplefreq_df["Sample.AltDepth"]
            after_values = caller_df["Sample.AltDepth"]
        
            if altdepth_kde_xmax is not None:
                
                before_values = before_values[before_values <= altdepth_kde_xmax]
                after_values = after_values[after_values <= altdepth_kde_xmax]
        
            plotted_any |= plot_kde(
                before_values,
                f"Before sample-frequency filter (n={len(before_samplefreq_df):,})"
            )
        
            plotted_any |= plot_kde(
                after_values,
                f"After sample-frequency filter (n={len(caller_df):,})"
            )
        
            if plotted_any:
                if altdepth_kde_xmax is not None:
                    plt.xlim(0, altdepth_kde_xmax)
        
                plt.xlabel("Sample.AltDepth")
                plt.ylabel("Density")
                plt.title(
                    f"{run_label} {set_name}: {caller_name} AltDepth KDE\n"
                    f"before vs after sample frequency <= {sample_freq_threshold:.0%}"
                )
                plt.legend(fontsize=8)
        
                out_png = figure_dir / (
                    f"{output_prefix}_{run_label}_{set_name}_{safe_name(caller_name)}"
                    f"_samplefreq_le_{threshold_label(sample_freq_threshold)}"
                    f"_altdepth_kde_before_vs_after_samplefreq.png"
                )
                save_plot(out_png)
            else:
                plt.close()
                print(f"Skipped before/after AltDepth KDE: {set_name}, {caller_name}")
        
            plot_vaf_density_by_altdepth(
                caller_df, caller_name, set_name, sample_freq_threshold, figure_dir
            )
        
            plot_altdepth_kde_by_altdepth(
                caller_df, caller_name, set_name, sample_freq_threshold, figure_dir
            )
        
            for altdepth_threshold in altdepth_thresholds:
                if altdepth_threshold is None:
                    subset = caller_df.copy()
                else:
                    subset = caller_df[caller_df["Sample.AltDepth"] >= altdepth_threshold].copy()
        
                summary_rows.append({
                    "analysis_set": set_name,
                    "caller": caller_name,
                    "sample_frequency_threshold": sample_freq_threshold,
                    "altdepth_threshold": "none" if altdepth_threshold is None else altdepth_threshold,
                    "rows_after_filters": len(subset),
                    "unique_variants_after_filters": subset.drop_duplicates(subset=variant_cols).shape[0],
                    "median_vaf": subset["Sample.AltFrac"].median(),
                    "median_altdepth": subset["Sample.AltDepth"].median(),
                    "max_sample_fraction_kept": subset["sample_fraction"].max(),
                })

## 7. Save summary table

This table gives the row count, unique variant count, median VAF, and median AltDepth after each filter combination.

In [ ]:
summary_df = pd.DataFrame(summary_rows)
summary_csv = output_dir / f"{output_prefix}_{run_label}_samplefreq_altdepth_threshold_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("Saved:", summary_csv)
summary_df